In [1]:
# ============================================================
# Anwendung des Linear Probes auf den gesamten Datensatz
# ============================================================
import os
import shutil
import torch
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import mobileclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

# --- Modell laden (gleich wie vorher) ---
weights_path = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"
model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=weights_path,
    device=device
)
model.eval()
print("Modell geladen: MobileCLIP-S1")

# --- Trainierten Linear Probe laden ---
clf = joblib.load("linear_probe_mobileclip_human_nonhuman.joblib")
print("Linear Probe geladen")

# --- Alle Bilder im Ordner finden (gleicher Ordner wie in deinem Original-Skript) ---
full_dataset_folder = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Indoor_Outdoor\final_class_indoor_outdoor_sorted\outdoor"
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

full_image_paths = []
for root, _, files in os.walk(full_dataset_folder):
    for fname in sorted(files):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTENSIONS:
            full_image_paths.append(os.path.join(root, fname))

print(f"Bilder gefunden: {len(full_image_paths)}")

# --- Embeddings extrahieren + direkt klassifizieren ---
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError):
        return None

results, confidences, valid_paths = [], [], []

for path in tqdm(full_image_paths, desc="Klassifiziere"):
    img = safe_open(path)
    if img is None:
        continue
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    embedding = feat.cpu().numpy()  # shape (1, dim)

    pred = clf.predict(embedding)[0]              # "human" oder "non_human"
    conf = clf.predict_proba(embedding).max()      # Konfidenz der Vorhersage

    results.append(pred)
    confidences.append(conf)
    valid_paths.append(path)

# --- Ergebnisse speichern ---
output_csv = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE.csv"

df_results = pd.DataFrame({
    "file_name": [os.path.basename(p) for p in valid_paths],
    "file_path": valid_paths,
    "pred_label": results,
    "confidence": confidences
})
df_results.to_csv(output_csv, index=False)

print(f"\n✅ Klassifikation abgeschlossen.")
print(f"Gespeichert: {output_csv}")
print("\nVerteilung im Gesamtdatensatz:")
print(df_results["pred_label"].value_counts())

# --- Optional: Bilder in Ordner sortieren (wie in deinem Original-Skript) ---
output_base = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE_vitl14_sorted"
categories = ["human", "non_human"]
sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in tqdm(zip(valid_paths, results), total=len(valid_paths), desc="Kopiere Bilder"):
    dest = os.path.join(sorted_dirs[pred], os.path.basename(path))
    if not os.path.exists(dest):
        shutil.copy(path, dest)

print("\nBilder sortiert:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)} → {sorted_dirs[cat]}")

C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lukas\miniconda3\envs\kilimanjaro\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Gerät: cuda
Modell geladen: MobileCLIP-S1
Linear Probe geladen
Bilder gefunden: 8219


Klassifiziere: 100%|███████████████████████████████████████████████████████████████| 8219/8219 [15:44<00:00,  8.70it/s]



✅ Klassifikation abgeschlossen.
Gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE.csv

Verteilung im Gesamtdatensatz:
pred_label
non_human    5094
human        3125
Name: count, dtype: int64


Kopiere Bilder: 100%|█████████████████████████████████████████████████████████████| 8219/8219 [00:22<00:00, 370.42it/s]


Bilder sortiert:
  human: 3125 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE_sorted\human
  non_human: 5094 → C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Human_non_Human\final_class_human_non_human_LINEARPROBE_sorted\non_human
